# 07 - End-to-End Demo: Detect then Classify
Run one image through YOLO detection and classify each detected solar-panel crop.


In [3]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

DETECTION_ROOT = PROJECT_ROOT / "rooftop-solar-panels-object-detection"
CLASSIFICATION_ROOT = PROJECT_ROOT / "rooftop-solar-panels-image-classification" / "Faulty_solar_panel"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DETECTION_ROOT exists:", DETECTION_ROOT.exists())
print("CLASSIFICATION_ROOT exists:", CLASSIFICATION_ROOT.exists())


PROJECT_ROOT: C:\Users\DigvijayYadav\Downloads\rooftop-solar-panel-dataset
DETECTION_ROOT exists: True
CLASSIFICATION_ROOT exists: True


In [4]:
from pipeline_utils import run_end_to_end_inference

# Auto-resolve latest detection weights
det_candidates = sorted(
    (PROJECT_ROOT / "artifacts/models").glob("det_yolov8n_cpu*/weights/best.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
det_weights = det_candidates[0] if det_candidates else None

# Auto-resolve classification checkpoint
cls_candidates = sorted(
    (PROJECT_ROOT / "artifacts/models").glob("cls_*_cpu.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
cls_ckpt = cls_candidates[0] if cls_candidates else None

# Pick a sample image from classification folders (all common image extensions), fallback to detection images
image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
class_images = [
    p for p in CLASSIFICATION_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in image_suffixes
]
det_images = [
    p for p in (DETECTION_ROOT / "images").rglob("*")
    if p.is_file() and p.suffix.lower() in image_suffixes
] if (DETECTION_ROOT / "images").exists() else []

sample_image = class_images[0] if class_images else (det_images[0] if det_images else None)

print("Resolved sample_image:", sample_image)
print("Resolved det_weights:", det_weights)
print("Resolved cls_ckpt:", cls_ckpt)

if sample_image is None:
    raise FileNotFoundError("No sample image found in classification or detection dataset folders")
if det_weights is None:
    raise FileNotFoundError("No detection best.pt found under artifacts/models/det_yolov8n_cpu*/weights/")
if cls_ckpt is None:
    raise FileNotFoundError("No classification checkpoint found under artifacts/models/cls_*_cpu.pt")

out_df = run_end_to_end_inference(
    sample_image=sample_image,
    det_weights=det_weights,
    cls_checkpoint=cls_ckpt,
    conf=0.25,
)
display(out_df)



Resolved sample_image: C:\Users\DigvijayYadav\Downloads\rooftop-solar-panel-dataset\rooftop-solar-panels-image-classification\Faulty_solar_panel\Bird-drop\Bird (1).jpeg
Resolved det_weights: C:\Users\DigvijayYadav\Downloads\rooftop-solar-panel-dataset\artifacts\models\det_yolov8n_cpu3\weights\best.pt
Resolved cls_ckpt: C:\Users\DigvijayYadav\Downloads\rooftop-solar-panel-dataset\artifacts\models\cls_mobilenetv3_small_cpu.pt


,image,bbox_id,det_confidence,predicted_condition,condition_probability,top3
0,C:\Users\DigvijayYadav\Downloads\rooftop-solar...,0,0.523595,Bird-drop,0.999451,"[{'class': 'Bird-drop', 'probability': 0.99945..."
1,C:\Users\DigvijayYadav\Downloads\rooftop-solar...,1,0.311828,Dusty,0.533954,"[{'class': 'Dusty', 'probability': 0.533953845..."
